### The purpose of this file is to run analysis on unintentional attack results to generate graphs

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

import os
import matplotlib.pyplot as plt
import seaborn as sns

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, PercentFormatter
from matplotlib.offsetbox import AnchoredText

In [ ]:
def plot_bleu_histogram(df, column_name, graph_title):
    """
    Create a histogram for specified BLEU score column with custom title.

    Parameters:
    df (pd.DataFrame): DataFrame containing BLEU scores
    column_name (str): The column name of the BLEU score
    graph_title (str): The title of the graph
    """
    fig, ax = plt.subplots(figsize=(12, 6))

    # Create histogram
    n, bins, patches = ax.hist(df[column_name], bins=30, color='skyblue', edgecolor='black')

    # Count occurrences of perfect BLEU score (1.0)
    perfect_count = len(df[df[column_name] == 1])

    # Highlight bin containing score 1
    if perfect_count > 0:
        for i in range(len(bins)-1):
            if bins[i] <= 1.0 <= bins[i+1]:
                patches[i].set_facecolor('red')
                ax.text(1, n[i], f'Exact Match : {perfect_count}', 
                        horizontalalignment='center', verticalalignment='bottom')
                break

    # Set labels and title
    ax.set_xlabel('BLEU Score Distribution for Membership Inference Attack')
    ax.set_ylabel('Count')
    ax.set_title(graph_title, pad=20)

    # Summary statistics
    mean_score = df[column_name].mean()
    median_score = df[column_name].median()
    std_score = df[column_name].std()

    stats_text = (f'Mean: {mean_score:.3f}\n'
                  f'Median: {median_score:.3f}\n'
                  f'Std Dev: {std_score:.3f}')

    # Legend at top-right corner
    ax.text(0.98, 0.98, stats_text,
            transform=ax.transAxes,
            verticalalignment='top',
            horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    # Remove grid lines
    ax.grid(False)

    plt.tight_layout()
    plt.show()


In [ ]:
def plot_bleu_histogram(df, column_name, graph_title):
    """
    Journal-style histogram for BLEU scores (membership inference), matched to the
    extraction-attack format: blue bars, % y-axis, fixed [0,1], hatch at BLEU=1.0,
    mean/median box only.
    """
    # Clean data
    data = pd.to_numeric(df[column_name], errors="coerce").dropna()
    n_total = len(data)
    if n_total == 0:
        raise ValueError("No numeric data found in the specified column.")

    # Fixed binning 0–1 (40 bins)
    edges = np.linspace(0, 1, 41)

    # Typography/layout (same as extraction)
    rc = {
        "font.family": "serif",
        "font.size": 10,        # title=12, ticks≈9 via relative sizes below
        "axes.titlesize": 12,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "figure.dpi": 300,
    }

    with plt.rc_context(rc):
        fig, ax = plt.subplots(figsize=(7, 4.25), constrained_layout=True, facecolor="white")

        # Histogram as percentage of samples per bin
        counts, edges, patches = ax.hist(
            data,
            bins=edges,
            weights=np.ones_like(data, dtype=float) / n_total,
            color="#1f77b4",      # blue bars
            edgecolor="#0e3a5e",
            linewidth=0.7,
        )

        # Highlight BLEU=1.0 bin + vertical reference line
        idx_1 = np.digitize([1.0], edges, right=False)[0] - 1
        if 0 <= idx_1 < len(patches):
            patches[idx_1].set_hatch("//")
            patches[idx_1].set_edgecolor("0.15")
        ax.axvline(1.0, linestyle="--", linewidth=1.2, color="0.25")

        # Labels & title
        ax.set_xlabel("BLEU score (4-gram)", labelpad=6)
        ax.set_ylabel("Percentage of Samples", labelpad=6)
        ax.set_title(graph_title, pad=12)

        # Ticks & limits
        ax.set_xlim(0, 1)
        ax.xaxis.set_major_locator(MultipleLocator(0.1))
        ax.xaxis.set_minor_locator(MultipleLocator(0.05))
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
        ax.yaxis.set_major_locator(MultipleLocator(0.10))   # 10%
        ax.yaxis.set_minor_locator(MultipleLocator(0.05))   # 5%

        # Clean look
        ax.grid(axis="y", linestyle="-", linewidth=0.5, alpha=0.35)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_linewidth(1.0)
        ax.spines["bottom"].set_linewidth(1.0)

        # Stats box: ONLY mean and median
        mean, median = data.mean(), data.median()
        stats = f"Mean {mean:.3f}\nMedian {median:.3f}"
        at = AnchoredText(stats, loc="upper left", prop={"size": 9}, frameon=True)
        at.patch.set_boxstyle("round,pad=0.3")
        at.patch.set_alpha(0.95)
        at.patch.set_edgecolor("0.7")
        ax.add_artist(at)

        # Exact-match annotation at BLEU=1.0
        n_exact = int(np.isclose(data, 1.0).sum())
        if n_exact > 0:
            pct = n_exact / n_total
            ax.annotate(
                f"Exact matches: {pct:.2%} (n={n_exact:,})",
                xy=(1.0, 1.0),
                xycoords=("data", "axes fraction"),
                xytext=(-6, -10),
                textcoords="offset points",
                ha="right",
                va="top",
                fontsize=9,
                bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="0.7", alpha=0.95),
            )

        plt.show()

In [ ]:
df_lm = pd.read_csv("/Users/anonymous_user/Downloads/experiment_results/exp4_attribute_analysis_results/step4_result_processing/step4_result/llama_150.csv")
df_sc_7b = pd.read_csv("/Users/anonymous_user/Downloads/experiment_results/exp4_attribute_analysis_results/step4_result_processing/step4_result/sc7b_150.csv")

In [ ]:
### only run result analysis on those two models

plot_bleu_histogram(df_lm, 'code_only_bleu_150', 'BLEU Score Distribution for Llama 150')

In [ ]:
plot_bleu_histogram(df_sc_7b, 'code_only_bleu_150', 'BLEU Score Distribution for SC7B 150')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, PercentFormatter
from matplotlib.offsetbox import AnchoredText

def plot_bleu_histogram(df, column_name, graph_title, *, threshold=0.85, total_bins=40, trim_right=True):
    """
    BLEU histogram (journal style):
    - Equal-width blue bins up to `threshold`; ONE aggregated blue bin (same width) for > threshold.
    - Y-axis shows percentage of samples.
    - Mean/Median box (upper-left). High-BLEU summary box (upper-right, inside frame).
    - If trim_right=True, x-axis ends at the aggregated bin so no empty right space is shown.
    """
    # Colors (single blue across all bars)
    bar_color = "#1f77b4"   # blue
    edge_color = "#0e3a5e"

    # Data
    data = pd.to_numeric(df[column_name], errors="coerce").dropna()
    n_total = len(data)
    if n_total == 0:
        raise ValueError("No numeric data found in the specified column.")

    # Fixed binning across [0,1]
    edges_full = np.linspace(0.0, 1.0, total_bins + 1)
    bin_width = edges_full[1] - edges_full[0]

    # Align threshold to nearest bin edge and compute left bins
    k = int(round(threshold / bin_width))
    thr_edge = k * bin_width
    k = min(max(k, 1), total_bins - 1)

    # Split data
    low_data = data[data <= thr_edge]
    high_mask = data > thr_edge
    high_count = int(high_mask.sum())
    high_prop = high_count / n_total
    n_exact = int(np.isclose(data, 1.0).sum())

    # Typography
    rc = {
        "font.family": "serif",
        "font.size": 10,
        "axes.titlesize": 12,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "figure.dpi": 300,
    }

    with plt.rc_context(rc):
        fig, ax = plt.subplots(figsize=(7, 4.25), constrained_layout=True, facecolor="white")

        # Left-side histogram (<= threshold) as percentages
        if len(low_data) > 0:
            low_edges = edges_full[:k+1]  # bins 0..k
            ax.hist(
                low_data,
                bins=low_edges,
                weights=np.ones(len(low_data), dtype=float) / n_total,
                color=bar_color,
                edgecolor=edge_color,
                linewidth=0.7,
            )

        # Single aggregated bin for > threshold — SAME color and width as others
        x_center = thr_edge + bin_width / 2.0
        if high_count > 0:
            ax.bar(
                x_center, high_prop,
                width=bin_width,
                color=bar_color,         # same blue
                edgecolor=edge_color,
                linewidth=0.7,
            )

        # Labels & title
        ax.set_xlabel("BLEU score", labelpad=6)
        ax.set_ylabel("Percentage of samples", labelpad=6)
        ax.set_title(graph_title, pad=12)

        # X range (trim at right edge of the aggregated bin if requested)
        x_right = thr_edge + bin_width if trim_right else 1.0
        ax.set_xlim(0, x_right)

        # Ticks & formatting
        ax.xaxis.set_major_locator(MultipleLocator(0.1))
        ax.xaxis.set_minor_locator(MultipleLocator(0.05))
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
        ax.yaxis.set_major_locator(MultipleLocator(0.10))
        ax.yaxis.set_minor_locator(MultipleLocator(0.05))

        # Clean look
        ax.grid(axis="y", linestyle="-", linewidth=0.5, alpha=0.35)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_linewidth(1.0)
        ax.spines["bottom"].set_linewidth(1.0)

        # Stats box (upper-left): Mean & Median only
        mean, median = data.mean(), data.median()
        stats = f"Mean {mean:.3f}\nMedian {median:.3f}"
        at_left = AnchoredText(stats, loc="upper left", prop={"size": 9}, frameon=True)
        at_left.patch.set_boxstyle("round,pad=0.3")
        at_left.patch.set_alpha(0.95)
        at_left.patch.set_edgecolor("0.7")
        ax.add_artist(at_left)

        # High-BLEU summary (upper-right, inside frame)
        if high_count > 0:
            label = f"High BLEU (>{thr_edge:.2f}): {high_prop:.2%} (n={high_count:,})"
            if n_exact > 0:
                label += f"\n(incl. exact: {n_exact:,})"
            at_right = AnchoredText(label, loc="upper right", prop={"size": 9}, frameon=True)
            at_right.patch.set_boxstyle("round,pad=0.3")
            at_right.patch.set_alpha(0.95)
            at_right.patch.set_edgecolor("0.7")
            ax.add_artist(at_right)

        plt.show()


In [ ]:
plot_bleu_histogram(df_sc_7b, 'code_only_bleu_150', 'Unintentional Attack Result for Starcoder2-7B')

In [ ]:
plot_bleu_histogram(df_lm, 'code_only_bleu_150', 'Unintentional Attack Result for Llama3 8B')

In [ ]:
df_lm.shape, df_sc_7b.shape